# PM3 — Tratamento de Dados PNAD Contínua (IBGE)

**Pré-requisitos:** `01_coleta.ipynb` e `02_silver.ipynb` já executados
(`dados/bronze/pnad_*.json` e `dados/silver/pnad_limpo.csv` existentes).

In [2]:
import sys, pathlib

_root = pathlib.Path.cwd()
if not (_root / "config.py").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [3]:
import sys, pathlib

_root = pathlib.Path.cwd()
if not (_root / "config.py").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from config import config

print(f"Bronze dir : {config.bronze_dir}")
print(f"Silver dir : {config.silver_dir}")

Bronze dir : c:\Users\gabri\OneDrive\Documentos\GitHub\mensal_3_data_cleaning\mensal_3\dados\bronze
Silver dir : c:\Users\gabri\OneDrive\Documentos\GitHub\mensal_3_data_cleaning\mensal_3\dados\silver


In [4]:
from silver import PnadNormalizer

print("=== Fase 3.1 — Normalizar PNAD ===")
pnad_norm = PnadNormalizer(config.bronze_dir, config.silver_dir)
df_pnad = pnad_norm.normalize()
df_pnad.head()

=== Fase 3.1 — Normalizar PNAD ===
  [pnad silver] 5,920 linhas → c:\Users\gabri\OneDrive\Documentos\GitHub\mensal_3_data_cleaning\mensal_3\dados\silver\pnad_limpo.csv


,tabela_id,uf_cod,uf_nome,periodo,sexo_cod,sexo,variavel_cod,variavel_nome,unidade_medida_cod,unidade_medida,valor,regiao_cod,regiao_nome,nivel_instrucao_cod,nivel_instrucao
0,4093,41,Paraná,202001,4,Masculino,1641,Pessoas de 14 anos ou mais de idade,1572,Mil pessoas,4529.0,NaN,NaN,NaN,NaN
1,4093,41,Paraná,202001,5,Feminino,1641,Pessoas de 14 anos ou mais de idade,1572,Mil pessoas,4701.0,NaN,NaN,NaN,NaN
2,4093,41,Paraná,202001,4,Masculino,4087,Coeficiente de variação - Pessoas de 14 anos ou mais de idade,2,%,0.7,NaN,NaN,NaN,NaN
3,4093,41,Paraná,202001,5,Feminino,4087,Coeficiente de variação - Pessoas de 14 anos ou mais de idade,2,%,0.7,NaN,NaN,NaN,NaN
4,4093,41,Paraná,202001,4,Masculino,4104,Distribuição percentual das pessoas de 14 anos ou mais de idade,2,%,49.1,NaN,NaN,NaN,NaN


In [5]:
import json

import pandas as pd
import plotly.express as px

pd.set_option("display.max_colwidth", 80)

## 5. Diagnóstico da qualidade dos dados

Análise feita a partir do **bronze** (JSON bruto retornado pela API SIDRA), antes da
normalização do `PnadNormalizer`, para expor os problemas reais da base original.

In [6]:
bronze = {}
for tabela in ["4093", "5436", "7322"]:
    bronze[tabela] = json.loads(
        (config.bronze_dir / f"pnad_{tabela}.json").read_text(encoding="utf-8")
    )
    print(f"pnad_{tabela}.json -> {len(bronze[tabela]) - 1:,} registros + 1 linha de cabeçalho")

print("\nLinha 0 (cabeçalho de metadados, tabela 4093):")
bronze["4093"][0]

pnad_4093.json -> 4,896 registros + 1 linha de cabeçalho
pnad_5436.json -> 768 registros + 1 linha de cabeçalho
pnad_7322.json -> 256 registros + 1 linha de cabeçalho

Linha 0 (cabeçalho de metadados, tabela 4093):


{'NC': 'Nível Territorial (Código)',
 'NN': 'Nível Territorial',
 'MC': 'Unidade de Medida (Código)',
 'MN': 'Unidade de Medida',
 'V': 'Valor',
 'D1C': 'Unidade da Federação (Código)',
 'D1N': 'Unidade da Federação',
 'D2C': 'Trimestre (Código)',
 'D2N': 'Trimestre',
 'D3C': 'Variável (Código)',
 'D3N': 'Variável',
 'D4C': 'Sexo (Código)',
 'D4N': 'Sexo'}

In [7]:
print("Linha 1 (primeiro registro de dados, tabela 4093):")
bronze["4093"][1]

Linha 1 (primeiro registro de dados, tabela 4093):


{'NC': '3',
 'NN': 'Unidade da Federação',
 'MC': '1572',
 'MN': 'Mil pessoas',
 'V': '4529',
 'D1C': '41',
 'D1N': 'Paraná',
 'D2C': '202001',
 'D2N': '1º trimestre 2020',
 'D3C': '1641',
 'D3N': 'Pessoas de 14 anos ou mais de idade',
 'D4C': '4',
 'D4N': 'Homens'}

**Problema 1 — cabeçalho de metadados na linha 0.** A API SIDRA retorna a linha 0 com
as *descrições* das colunas (`D1C`/`D1N`, `D2C`/`D2N`, ..., `MC`/`MN`, `V`), e a partir
da linha 1 os dados propriamente ditos — usando os mesmos nomes de chave. Se essa
linha 0 não for descartada e usada apenas para mapear os nomes das colunas, ela
aparece como um registro inválido no dataset (todas as colunas como texto
descritivo). O `PnadNormalizer` já trata isso: usa `dados[0]` apenas para construir
`_build_col_map` e descarta a linha do dataframe final.

**Problema 2 — códigos numéricos vs. nomes redundantes.** Cada dimensão vem em pares
`DnC` (código) / `DnN` (nome) — ex.: `D1C="41"` / `D1N="Paraná"`,
`D4C="4"` / `D4N="Homens"`. Isso duplica a informação e precisa ser resolvido na
seleção de colunas (seção 7).

**Problema 3 — `Trimestre`/`Ano` (`D2C`) como código de texto, não como data.** Para
4093/5436, `D2C` vem no formato `AAAASS` (ex.: `"202001"` = 1º trimestre de 2020); para
7322, `D2C` é só o ano (`"2021"`). Precisa virar `ano`/`trimestre` (seção 8).

In [8]:
_AUSENTES_IBGE = {"-", "...", "X", "x", "C", ""}

print("Códigos de ausência do IBGE na coluna 'V' (valor bruto):\n")
for tabela, dados in bronze.items():
    df_raw = pd.DataFrame(dados[1:])
    n_ausentes = df_raw["V"].isin(_AUSENTES_IBGE).sum()
    n_virgula = df_raw["V"].str.contains(",", regex=False).sum()
    print(f"  tabela {tabela}: {n_ausentes:>5,} / {len(df_raw):,} valores ausentes"
          f" | {n_virgula} valores com vírgula decimal")
    if n_ausentes:
        print("   ", dict(df_raw.loc[df_raw['V'].isin(_AUSENTES_IBGE), 'V'].value_counts()))

Códigos de ausência do IBGE na coluna 'V' (valor bruto):

  tabela 4093: 1,632 / 4,896 valores ausentes | 0 valores com vírgula decimal
    {'...': np.int64(1632)}
  tabela 5436:     0 / 768 valores ausentes | 0 valores com vírgula decimal
  tabela 7322:     8 / 256 valores ausentes | 0 valores com vírgula decimal
    {'-': np.int64(8)}


**Problema 4 — valores ausentes codificados pelo IBGE.** A tabela 4093 tem **1.632**
valores `"..."` (≈33% das suas 4.896 linhas) e a tabela 7322 tem **8** valores `"-"`
(≈3% das 256 linhas) — total de **1.640** valores ausentes na coluna `valor`. O
`PnadNormalizer` converte todos os códigos `{-, ..., X, x, C, ""}` para `NaN` via
`pd.to_numeric(..., errors="coerce")`. Esses ausentes são tratados na seção 9.

> **Nota:** ao contrário do esperado inicialmente, **nenhum** valor numérico desta
> coleta usa vírgula como separador decimal (todos já vêm com ponto, ex. `"49.1"`).
> A normalização `valor_str.str.replace(",", ".")` no `PnadNormalizer` é defensiva
> (não tem efeito nos dados atuais, mas protege contra outras tabelas SIDRA que usam
> vírgula).

In [9]:
print("Problema 5 — dimensão 'Variável' (D3) empilhada na mesma coluna 'V':\n")
for tabela, dados in bronze.items():
    df_raw = pd.DataFrame(dados[1:])
    n_vars = df_raw["D3N"].nunique()
    unidades = df_raw["MN"].unique()
    print(f"  tabela {tabela}: {n_vars} variáveis distintas | unidades de medida: {list(unidades)}")

print("\nExemplo — tabela 4093, variáveis e unidades (5 primeiras):")
df_4093_raw = pd.DataFrame(bronze["4093"][1:])
df_4093_raw[["D3N", "MN"]].drop_duplicates().head()

Problema 5 — dimensão 'Variável' (D3) empilhada na mesma coluna 'V':

  tabela 4093: 34 variáveis distintas | unidades de medida: ['Mil pessoas', '%']
  tabela 5436: 8 variáveis distintas | unidades de medida: ['Reais', '%']
  tabela 7322: 4 variáveis distintas | unidades de medida: ['Mil pessoas', '%']

Exemplo — tabela 4093, variáveis e unidades (5 primeiras):


,D3N,MN
0,Pessoas de 14 anos ou mais de idade,Mil pessoas
2,Coeficiente de variação - Pessoas de 14 anos ou mais de idade,%
4,Distribuição percentual das pessoas de 14 anos ou mais de idade,%
6,Coeficiente de variação - Distribuição percentual das pessoas de 14 anos ou ...,%
8,"Pessoas de 14 anos ou mais de idade, na força de trabalho, na semana de refe...",Mil pessoas


**Problema 5 (principal) — múltiplos indicadores empilhados na mesma coluna `valor`,
com unidades diferentes.** Cada tabela SIDRA traz a dimensão **"Variável" (D3)** com
dezenas de indicadores diferentes (contagens em "Mil pessoas", taxas em "%",
rendimentos em "Reais" e "Coeficientes de variação" — uma medida de precisão
estatística, não um indicador de negócio):

- **4093** → 34 variáveis (ocupados, desocupados, taxa de informalidade, taxa de
  participação etc., cada uma com seu "Coeficiente de variação - ...").
- **5436** → 8 variáveis (rendimento habitual/efetivo × trabalho principal/todos os
  trabalhos, e respectivos coeficientes de variação).
- **7322** → 4 variáveis (pessoas por nível de instrução, distribuição percentual, e
  coeficientes de variação).

Sem isolar por `variavel_nome` (e `unidade_medida`), qualquer estatística ou gráfico
sobre `valor` mistura grandezas incompatíveis (pessoas, %, R$, coeficientes de
variação). O `PnadNormalizer` já preserva `variavel_cod`/`variavel_nome` e
`unidade_medida_cod`/`unidade_medida` como colunas próprias — a correção definitiva é
**filtrar por `variavel_nome`** na seleção de dados (seção 7).

**Problema 6 — mistura de unidades entre `tabela_id`.** Mesmo após filtrar por
`variavel_nome`, `tabela_id == "4093"` (pessoas/%) e `tabela_id == "5436"` (R$) usam
escalas totalmente diferentes — qualquer comparação de `valor` entre tabelas sem
filtrar por `tabela_id` é inválida.

**Problema 7 — recorte etário inconsistente entre tabelas.** As tabelas 4093 e 5436
cobrem pessoas de **14 anos ou mais**, enquanto a tabela 7322 cobre pessoas de
**10 anos ou mais**. Comparações entre `tabela_id == "7322"` e as demais (ex.:
população total) devem registrar essa diferença de público-alvo — não são diretamente
somáveis/comparáveis.

**Problema 8 — tabela 7322 coletada em nível geográfico diferente (N2/Região, não
N3/UF).** Suas linhas têm `regiao_cod`/`regiao_nome` preenchidos e
`uf_cod`/`uf_nome` vazios (`NaN`), enquanto 4093/5436 são o oposto.

In [10]:
print("Problema 9 — períodos ausentes inteiros (não apenas valores '...'):\n")
for tabela in ["4093", "5436"]:
    df_raw = pd.DataFrame(bronze[tabela][1:])
    periodos_presentes = sorted(df_raw["D2C"].unique())
    todos_periodos = [f"{ano}{tri:02d}" for ano in range(2020, 2026) for tri in range(1, 5)]
    faltando = [p for p in todos_periodos if p not in periodos_presentes]
    print(f"  tabela {tabela}: {len(periodos_presentes)}/{len(todos_periodos)} trimestres "
          f"presentes na resposta da API")
    if faltando:
        print(f"    trimestres ausentes: {faltando}")

Problema 9 — períodos ausentes inteiros (não apenas valores '...'):

  tabela 4093: 24/24 trimestres presentes na resposta da API
  tabela 5436: 16/24 trimestres presentes na resposta da API
    trimestres ausentes: ['202002', '202003', '202004', '202101', '202102', '202103', '202104', '202201']


**Problema 9 — trimestres inteiros ausentes na resposta da API (tabela 5436).** A
tabela 5436 retorna apenas **16 dos 24 trimestres** esperados (2020T1 a 2025T4) — os
8 trimestres de **2020T2 a 2022T1** simplesmente não aparecem na resposta da API
SIDRA (não é um valor `"..."`, a linha não existe). A tabela 4093 está completa
(24/24 trimestres, com `"..."` em alguns indicadores). Esse "buraco" temporal em
5436 deve ser registrado no relatório (seção 9) e aparecerá como uma lacuna no
gráfico de evolução temporal (seção 6.6).

In [11]:
tabela_problemas = pd.DataFrame([
    {
        "problema": "Cabeçalho de metadados na linha 0",
        "exemplo": "dados[0] = {'D1C': 'Unidade da Federação (Código)', ...}",
        "tratamento": "Usado apenas para mapear nomes de colunas (_build_col_map); descartado do dataframe final",
    },
    {
        "problema": "Códigos vs. nomes redundantes (DnC/DnN)",
        "exemplo": "uf_cod=41 / uf_nome='Paraná', sexo_cod=4 / sexo='Masculino'",
        "tratamento": "Manter nomes para leitura/EDA; remover códigos redundantes na seleção (seção 7)",
    },
    {
        "problema": "Período como código de texto (AAAASS / AAAA)",
        "exemplo": "periodo=202001 (4093/5436), periodo=2021 (7322)",
        "tratamento": "Derivar colunas ano (int) e trimestre (int, quando aplicável) na limpeza (seção 8)",
    },
    {
        "problema": "Valores ausentes codificados pelo IBGE",
        "exemplo": "valor_str = '...' (4093, 1.632x) ou '-' (7322, 8x)",
        "tratamento": "Convertidos para NaN via pd.to_numeric(errors='coerce'); tratados na seção 9",
    },
    {
        "problema": "Múltiplos indicadores empilhados em 'valor' (dimensão Variável)",
        "exemplo": "tabela 4093 tem 34 'variavel_nome' distintos com unidades Mil pessoas/%/Reais",
        "tratamento": "Filtrar por variavel_nome (4 indicadores de interesse) na seleção (seção 7) — problema principal",
    },
    {
        "problema": "Mistura de unidades entre tabela_id",
        "exemplo": "4093/7322 = Mil pessoas/%, 5436 = Reais",
        "tratamento": "Sempre agrupar/comparar 'valor' por (tabela_id, variavel_nome, unidade_medida)",
    },
    {
        "problema": "Recorte etário inconsistente entre tabelas",
        "exemplo": "4093/5436 = 14+ anos, 7322 = 10+ anos",
        "tratamento": "Documentar no relatório; não somar/comparar população entre 7322 e as demais",
    },
    {
        "problema": "Nível geográfico distinto (UF vs. Região)",
        "exemplo": "7322: uf_nome=NaN, regiao_nome='Sul'; 4093/5436: uf_nome preenchido, regiao_nome=NaN",
        "tratamento": "Tratar uf_nome/regiao_nome como 'Não se aplica' conforme a tabela (seção 9)",
    },
    {
        "problema": "Trimestres inteiros ausentes na resposta da API (5436)",
        "exemplo": "5436 tem 16/24 trimestres (faltam 2020T2 a 2022T1)",
        "tratamento": "Registrar lacuna temporal no relatório; não interpolar sem justificativa (seção 9)",
    },
])
tabela_problemas

,problema,exemplo,tratamento
0,Cabeçalho de metadados na linha 0,"dados[0] = {'D1C': 'Unidade da Federação (Código)', ...}",Usado apenas para mapear nomes de colunas (_build_col_map); descartado do da...
1,Códigos vs. nomes redundantes (DnC/DnN),"uf_cod=41 / uf_nome='Paraná', sexo_cod=4 / sexo='Masculino'",Manter nomes para leitura/EDA; remover códigos redundantes na seleção (seção 7)
2,Período como código de texto (AAAASS / AAAA),"periodo=202001 (4093/5436), periodo=2021 (7322)","Derivar colunas ano (int) e trimestre (int, quando aplicável) na limpeza (se..."
3,Valores ausentes codificados pelo IBGE,"valor_str = '...' (4093, 1.632x) ou '-' (7322, 8x)",Convertidos para NaN via pd.to_numeric(errors='coerce'); tratados na seção 9
4,Múltiplos indicadores empilhados em 'valor' (dimensão Variável),tabela 4093 tem 34 'variavel_nome' distintos com unidades Mil pessoas/%/Reais,Filtrar por variavel_nome (4 indicadores de interesse) na seleção (seção 7) ...
5,Mistura de unidades entre tabela_id,"4093/7322 = Mil pessoas/%, 5436 = Reais","Sempre agrupar/comparar 'valor' por (tabela_id, variavel_nome, unidade_medida)"
6,Recorte etário inconsistente entre tabelas,"4093/5436 = 14+ anos, 7322 = 10+ anos",Documentar no relatório; não somar/comparar população entre 7322 e as demais
7,Nível geográfico distinto (UF vs. Região),"7322: uf_nome=NaN, regiao_nome='Sul'; 4093/5436: uf_nome preenchido, regiao_...",Tratar uf_nome/regiao_nome como 'Não se aplica' conforme a tabela (seção 9)
8,Trimestres inteiros ausentes na resposta da API (5436),5436 tem 16/24 trimestres (faltam 2020T2 a 2022T1),Registrar lacuna temporal no relatório; não interpolar sem justificativa (se...


## 6. Análise Exploratória de Dados
A partir daqui trabalhamos com o **silver** (`pnad_limpo.csv`, 5.920 linhas × 15
colunas), já com `valor` numérico e `variavel_nome`/`unidade_medida` preservados.

### 6.1 Visão estrutural geral

In [12]:
df = pd.read_csv(config.silver_dir / "pnad_limpo.csv")

print(f"shape: {df.shape}")
df.info()

shape: (5920, 15)
<class 'pandas.DataFrame'>
RangeIndex: 5920 entries, 0 to 5919
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   tabela_id            5920 non-null   int64  
 1   uf_cod               5664 non-null   float64
 2   uf_nome              5664 non-null   str    
 3   periodo              5920 non-null   int64  
 4   sexo_cod             5920 non-null   int64  
 5   sexo                 5920 non-null   str    
 6   variavel_cod         5920 non-null   int64  
 7   variavel_nome        5920 non-null   str    
 8   unidade_medida_cod   5920 non-null   int64  
 9   unidade_medida       5920 non-null   str    
 10  valor                4280 non-null   float64
 11  regiao_cod           256 non-null    float64
 12  regiao_nome          256 non-null    str    
 13  nivel_instrucao_cod  256 non-null    float64
 14  nivel_instrucao      256 non-null    str    
dtypes: float64(4), int64(5), str(6)

In [13]:
# Valores ausentes por coluna (visão geral, antes de qualquer tratamento)
df.isna().sum().to_frame("n_ausentes").assign(pct=lambda x: (100 * x["n_ausentes"] / len(df)).round(1))

,n_ausentes,pct
tabela_id,0,0.0
uf_cod,256,4.3
uf_nome,256,4.3
periodo,0,0.0
sexo_cod,0,0.0
sexo,0,0.0
variavel_cod,0,0.0
variavel_nome,0,0.0
unidade_medida_cod,0,0.0
unidade_medida,0,0.0


### 6.2 Frequência das variáveis categóricas

In [14]:
print("tabela_id:")
print(df["tabela_id"].value_counts(), "\n")

print("sexo:")
print(df["sexo"].value_counts(), "\n")

print("uf_nome (NaN = registros da tabela 7322, nível Região):")
print(df["uf_nome"].value_counts(dropna=False), "\n")

print("regiao_nome (NaN = registros das tabelas 4093/5436, nível UF):")
print(df["regiao_nome"].value_counts(dropna=False), "\n")

print("nivel_instrucao (NaN = tabelas 4093/5436, sem essa classificação):")
print(df["nivel_instrucao"].value_counts(dropna=False))

tabela_id:
tabela_id
4093    4896
5436     768
7322     256
Name: count, dtype: int64 

sexo:
sexo
Masculino    2960
Feminino     2960
Name: count, dtype: int64 

uf_nome (NaN = registros da tabela 7322, nível Região):
uf_nome
Paraná               1888
Santa Catarina       1888
Rio Grande do Sul    1888
NaN                   256
Name: count, dtype: int64 

regiao_nome (NaN = registros das tabelas 4093/5436, nível UF):
regiao_nome
NaN    5664
Sul     256
Name: count, dtype: int64 

nivel_instrucao (NaN = tabelas 4093/5436, sem essa classificação):
nivel_instrucao
NaN                       5664
Total                       32
Sem instrução               32
Fundamental incompleto      32
Fundamental completo        32
Médio incompleto            32
Médio completo              32
Superior incompleto         32
Superior completo           32
Name: count, dtype: int64


### 6.3 Recorte para análise de `valor`

Conforme o **Problema 5** (seção 5), `valor` só é analisável depois de filtrar por
`variavel_nome`. Para a EDA a seguir, usamos os **4 indicadores de interesse** que
serão formalizados na seleção de dados (seção 7) — um indicador-chave por tabela,
cobrindo força de trabalho, informalidade, rendimento e nível de instrução.

In [15]:
VARIAVEIS_FOCO = [
    "Pessoas de 14 anos ou mais de idade ocupadas na semana de referência",
    "Taxa de informalidade das pessoas de 14 anos ou mais de idade ocupadas na semana de referência",
    "Rendimento médio mensal real das pessoas de 14 anos ou mais de idade ocupadas na semana de"
    " referência com rendimento de trabalho, habitualmente recebido no trabalho principal",
    "Pessoas de 10 anos ou mais de idade",
]

df_eda = df[df["variavel_nome"].isin(VARIAVEIS_FOCO)].copy()
print(f"df_eda: {df_eda.shape[0]} linhas (de {df.shape[0]} no silver completo)")
df_eda.groupby(["tabela_id", "variavel_nome", "unidade_medida"]).size().to_frame("n_linhas")

df_eda: 448 linhas (de 5920 no silver completo)


n_linhas
tabela_id variavel_nome                                                                    unidade_medida          
4093      Pessoas de 14 anos ou mais de idade ocupadas na semana de referência             Mil pessoas          144
          Taxa de informalidade das pessoas de 14 anos ou mais de idade ocupadas na sem... %                    144
5436      Rendimento médio mensal real das pessoas de 14 anos ou mais de idade ocupadas... Reais                 96
7322      Pessoas de 10 anos ou mais de idade                                              Mil pessoas           64

### 6.4 Estatísticas descritivas de `valor` por indicador

In [16]:
(
    df_eda.groupby(["tabela_id", "unidade_medida"])["valor"]
    .describe()
    .round(2)
)

count     mean      std     min      25%     50%  \
tabela_id unidade_medida                                                     
4093      %                96.0    29.88     2.91    23.8    27.15    31.0   
          Mil pessoas      96.0  2673.71   507.48  1668.0  2386.75  2583.5   
5436      Reais            96.0  3572.64   553.95  2654.0  3067.50  3601.5   
7322      Mil pessoas      64.0  3336.12  4024.58   303.0   875.00  1617.5   

                              75%      max  
tabela_id unidade_medida                    
4093      %                 32.30     33.9  
          Mil pessoas     3191.25   3474.0  
5436      Reais           3993.00   4676.0  
7322      Mil pessoas     3884.50  13755.0

### 6.5 Gráfico de barras — rendimento médio por UF e sexo

Rendimento médio mensal real (R$, tabela 5436), médio ao longo de todo o período
2020–2025, por UF e sexo.

In [17]:
df_rendimento = df_eda[df_eda["tabela_id"] == 5436]

rendimento_uf_sexo = (
    df_rendimento.groupby(["uf_nome", "sexo"])["valor"].mean().reset_index()
)

fig = px.bar(
    rendimento_uf_sexo,
    x="uf_nome",
    y="valor",
    color="sexo",
    barmode="group",
    text_auto=".0f",
    title="Rendimento médio mensal real (R$) por UF e sexo — média 2020-2025",
    labels={"uf_nome": "UF", "valor": "Rendimento médio (R$)", "sexo": "Sexo"},
)
fig.show()

**Interpretação:** em todos os três estados, o rendimento médio dos homens é
nitidamente superior ao das mulheres — Paraná (R$ 3.997 vs. R$ 2.984, gap de ≈34%),
Rio Grande do Sul (R$ 4.018 vs. R$ 3.058, gap de ≈31%) e Santa Catarina (R$ 4.182 vs.
R$ 3.198, gap de ≈31%). Santa Catarina tem o maior rendimento médio para ambos os
sexos, mas o gap percentual é semelhante entre os três estados — indicando que a
desigualdade de rendimento por sexo é um padrão regional, não específico de uma UF.
Esse "gap salarial" será formalizado como `gap_salarial_pct` na seção 15 (feature
engineering).

### 6.6 Gráfico de linha — evolução temporal do rendimento

Evolução trimestral do rendimento médio mensal real (R$, tabela 5436), por sexo,
agregando as três UFs.

In [18]:
# periodo no formato AAAASS -> rótulo "AAAA-Tn" para o eixo temporal
df_rendimento = df_rendimento.copy()
df_rendimento["ano"] = df_rendimento["periodo"] // 100
df_rendimento["trimestre"] = df_rendimento["periodo"] % 100
df_rendimento["periodo_label"] = (
    df_rendimento["ano"].astype(str) + "-T" + df_rendimento["trimestre"].astype(str)
)

evolucao = (
    df_rendimento.groupby(["periodo", "periodo_label", "sexo"])["valor"]
    .mean()
    .reset_index()
    .sort_values("periodo")
)

fig = px.line(
    evolucao,
    x="periodo_label",
    y="valor",
    color="sexo",
    markers=True,
    title="Evolução trimestral do rendimento médio mensal real (R$) por sexo — PR/SC/RS",
    labels={"periodo_label": "Trimestre", "valor": "Rendimento médio (R$)", "sexo": "Sexo"},
)
fig.show()

**Interpretação:** observa-se uma lacuna entre 2020T1 e 2022T2 — reflexo do
**Problema 9** (a tabela 5436 não retorna dados para 2020T2–2022T1). A partir de
2022T2, ambas as séries (homens e mulheres) mostram tendência de leve crescimento
real do rendimento até 2025T4, com a linha dos homens consistentemente acima da das
mulheres em todo o período observado — o gap salarial é persistente ao longo do
tempo, oscilando entre ≈30% e ≈35% (calculado a partir do `groupby` por período/sexo),
sem sinal de fechamento estrutural no horizonte analisado.

### 6.7 Boxplot — distribuição do rendimento por sexo e por UF

In [19]:
fig = px.box(
    df_rendimento,
    x="uf_nome",
    y="valor",
    color="sexo",
    title="Distribuição do rendimento médio mensal real (R$) por UF e sexo",
    labels={"uf_nome": "UF", "valor": "Rendimento médio (R$)", "sexo": "Sexo"},
)
fig.show()

**Interpretação:** os boxplots de homens ficam visivelmente acima dos de mulheres em
todas as UFs, com pouca sobreposição entre as caixas — reforçando que o gap salarial
não é causado por alguns trimestres atípicos, mas é uma diferença sistemática ao
longo de toda a série. As caixas de Santa Catarina estão deslocadas para cima em
relação a Paraná e Rio Grande do Sul (rendimentos mais altos), mas a amplitude
(variabilidade) é semelhante entre as três UFs, sem outliers extremos visíveis — algo
que será confirmado de forma mais rigorosa via IQR na seção 10.

### 6.8 Histograma — distribuição do rendimento

In [20]:
fig = px.histogram(
    df_rendimento,
    x="valor",
    color="sexo",
    nbins=20,
    barmode="overlay",
    opacity=0.7,
    title="Distribuição do rendimento médio mensal real (R$) — Mas. vs. Fem.",
    labels={"valor": "Rendimento médio (R$)", "sexo": "Sexo"},
)
fig.show()

**Interpretação:** as duas distribuições são unimodais e relativamente concentradas
(feminino entre R$ 2.654–3.621, masculino entre R$ 3.582–4.676), com a distribuição
masculina deslocada para a direita em relação à feminina e **praticamente sem
sobreposição** — visualização que complementa o boxplot, evidenciando que o gap
salarial se manifesta como um deslocamento de toda a distribuição, não apenas da
média.

### 6.9 Heatmap de correlação entre indicadores

Para correlacionar indicadores de tabelas/unidades diferentes, primeiro pivotamos
`df_eda` (tabelas 4093 e 5436, que compartilham `uf_nome`/`periodo`/`sexo`) para uma
linha por (UF, período, sexo) com uma coluna por indicador.

In [21]:
ROTULOS_INDICADOR = {
    "Pessoas de 14 anos ou mais de idade ocupadas na semana de referência": "ocupados_mil",
    "Taxa de informalidade das pessoas de 14 anos ou mais de idade ocupadas na semana de referência": "taxa_informalidade_pct",
    "Rendimento médio mensal real das pessoas de 14 anos ou mais de idade ocupadas na semana de"
    " referência com rendimento de trabalho, habitualmente recebido no trabalho principal": "rendimento_medio_r$",
}

df_4093_5436 = df_eda[df_eda["tabela_id"].isin([4093, 5436])].copy()
df_4093_5436["indicador"] = df_4093_5436["variavel_nome"].map(ROTULOS_INDICADOR)

df_pivot = df_4093_5436.pivot_table(
    index=["uf_nome", "periodo", "sexo"], columns="indicador", values="valor"
).reset_index()

corr = df_pivot[list(ROTULOS_INDICADOR.values())].corr().round(2)
corr

indicador,ocupados_mil,taxa_informalidade_pct,rendimento_medio_r$
indicador,,,
ocupados_mil,1.00,0.86,0.49
taxa_informalidade_pct,0.86,1.00,0.07
rendimento_medio_r$,0.49,0.07,1.00


In [22]:
fig = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Correlação entre indicadores (ocupados, taxa de informalidade, rendimento)",
)
fig.show()

**Interpretação:** `ocupados_mil` e `taxa_informalidade_pct` têm correlação forte
(0.86) — ambos crescem ao longo da série, possivelmente refletindo a recuperação do
mercado de trabalho pós-pandemia acompanhada de aumento da informalidade (mais
correlação temporal conjunta do que causalidade direta). `rendimento_medio_r$` tem
correlação moderada com `ocupados_mil` (0.49) e correlação muito fraca com
`taxa_informalidade_pct` (0.07) — sugerindo que o nível de rendimento médio não está
diretamente associado ao grau de informalidade nesta amostra (PR/SC/RS,
2020–2025).

### 6.10 Análise temporal — taxa de informalidade (tabela 4093)

Diferente de 5436, a tabela 4093 está completa nos 24 trimestres (2020T1–2025T4),
mas com valores `...`/`NaN` em alguns trimestres (Problema 4).

In [23]:
df_informalidade = df_eda[
    df_eda["variavel_nome"]
    == "Taxa de informalidade das pessoas de 14 anos ou mais de idade ocupadas na semana de referência"
].copy()
df_informalidade["ano"] = df_informalidade["periodo"] // 100
df_informalidade["trimestre"] = df_informalidade["periodo"] % 100
df_informalidade["periodo_label"] = (
    df_informalidade["ano"].astype(str) + "-T" + df_informalidade["trimestre"].astype(str)
)

evolucao_informalidade = (
    df_informalidade.groupby(["periodo", "periodo_label", "sexo"])["valor"]
    .mean()
    .reset_index()
    .sort_values("periodo")
)

fig = px.line(
    evolucao_informalidade,
    x="periodo_label",
    y="valor",
    color="sexo",
    markers=True,
    title="Evolução trimestral da taxa de informalidade (%) por sexo — PR/SC/RS",
    labels={"periodo_label": "Trimestre", "valor": "Taxa de informalidade (%)", "sexo": "Sexo"},
)
fig.show()

print(f"\nTrimestres com valor ausente (NaN): "
      f"{df_informalidade['valor'].isna().sum()} de {len(df_informalidade)}")


Trimestres com valor ausente (NaN): 48 de 144


**Interpretação:** 48 das 144 linhas (33%) têm `valor` ausente (`NaN`), todas
concentradas em **2020T2–2022T1** — exatamente o mesmo intervalo em que a tabela 5436
não retorna dados (Problema 9). Isso reforça a hipótese de que esse período reflete
uma limitação real de divulgação do IBGE para PR/SC/RS durante a fase mais aguda da
pandemia (provável undercoverage da coleta por telefone), e não um problema do
pipeline de coleta. Nos trimestres com dado disponível, a taxa de informalidade
feminina e masculina seguem padrões semelhantes, sem um gap tão marcante quanto o
observado no rendimento.

### 6.11 População da Região Sul por nível de instrução e sexo (tabela 7322)

In [24]:
df_instrucao = df_eda[
    (df_eda["tabela_id"] == 7322) & (df_eda["nivel_instrucao"] != "Total")
].copy()

instrucao_media = (
    df_instrucao.groupby(["nivel_instrucao", "sexo"])["valor"].mean().reset_index()
)

ordem_instrucao = [
    "Sem instrução", "Fundamental incompleto", "Fundamental completo",
    "Médio incompleto", "Médio completo", "Superior incompleto", "Superior completo",
]

fig = px.bar(
    instrucao_media,
    x="nivel_instrucao",
    y="valor",
    color="sexo",
    barmode="group",
    category_orders={"nivel_instrucao": ordem_instrucao},
    title="População (Mil pessoas, 10+ anos) da Região Sul por nível de instrução e sexo — média 2021-2024",
    labels={"nivel_instrucao": "Nível de instrução", "valor": "População (Mil pessoas)", "sexo": "Sexo"},
)
fig.update_xaxes(tickangle=30)
fig.show()

**Interpretação:** o "Fundamental incompleto" é o nível de instrução mais numeroso
para ambos os sexos (≈1.027 mil mulheres, ≈1.048 mil homens), seguido de "Médio
completo". O contraste mais notável aparece no "Superior completo": **mulheres
superam homens** (≈650 mil vs. ≈483 mil) — um padrão consistente com dados
educacionais nacionais (maior conclusão do ensino superior por mulheres). Combinado
com o gráfico de rendimento (seção 6.5), isso sugere que o gap salarial **não é
explicado por escolaridade** — mulheres têm, em média, mais anos de estudo, mas
recebem rendimentos menores.

---

## 7. Seleção dos dados 
### 7.1 Filtro de linhas — `variavel_nome`

Mantemos apenas os 4 indicadores de interesse já usados na EDA (`VARIAVEIS_FOCO`),
descartando todas as linhas de "Coeficiente de variação - ..." e "Distribuição
percentual - ..." (não são indicadores de negócio, são medidas de precisão
estatística do IBGE — ver Problema 5, seção 5).

### 7.2 Seleção de colunas

- **Mantidas:** `tabela_id`, `uf_nome`, `regiao_nome`, `periodo`, `sexo`,
  `nivel_instrucao`, `variavel_nome`, `unidade_medida`, `valor`.
- **Removidas (redundantes com a versão em texto):** `uf_cod`, `sexo_cod`,
  `regiao_cod`, `variavel_cod`, `nivel_instrucao_cod`, `unidade_medida_cod`.
- `periodo` é mantido nesta etapa e será decomposto em `ano`/`trimestre` na
  limpeza (seção 8).

`df_sel` continua combinando `tabela_id` 4093/5436/7322 até a seção 16, já que a
limpeza, o tratamento de ausentes/outliers e a normalização (seções 8–13) operam
por grupo `(tabela_id, variavel_nome)` e valem igualmente para as três tabelas. A
separação em dois datasets finais — **principal** (4093+5436, gap salarial) e de
**contexto** (7322, escolaridade) — só acontece na consolidação (seção 16), porque
é só ali que a diferença de granularidade entre as tabelas (UF×trimestre vs.
Região×ano) passa a importar.


In [25]:
COLUNAS_SELECIONADAS = [
    "tabela_id", "uf_nome", "regiao_nome", "periodo", "sexo",
    "nivel_instrucao", "variavel_nome", "unidade_medida", "valor",
]

df_sel = df[df["variavel_nome"].isin(VARIAVEIS_FOCO)][COLUNAS_SELECIONADAS].copy()

print(f"df_sel: {df_sel.shape[0]} linhas x {df_sel.shape[1]} colunas")
print(f"(de {df.shape[0]} linhas x {df.shape[1]} colunas no silver completo)")
df_sel.head()

df_sel: 448 linhas x 9 colunas
(de 5920 linhas x 15 colunas no silver completo)


,tabela_id,uf_nome,regiao_nome,periodo,sexo,nivel_instrucao,variavel_nome,unidade_medida,valor
16,4093,Paraná,NaN,202001,Masculino,NaN,Pessoas de 14 anos ou mais de idade ocupadas na semana de referência,Mil pessoas,3210.0
17,4093,Paraná,NaN,202001,Feminino,NaN,Pessoas de 14 anos ou mais de idade ocupadas na semana de referência,Mil pessoas,2374.0
56,4093,Paraná,NaN,202001,Masculino,NaN,Taxa de informalidade das pessoas de 14 anos ou mais de idade ocupadas na se...,%,32.4
57,4093,Paraná,NaN,202001,Feminino,NaN,Taxa de informalidade das pessoas de 14 anos ou mais de idade ocupadas na se...,%,30.8
84,4093,Paraná,NaN,202002,Masculino,NaN,Pessoas de 14 anos ou mais de idade ocupadas na semana de referência,Mil pessoas,NaN


### 7.3 Filtro de linhas — registros com `valor` ausente

Por ora **mantemos** as linhas com `valor` ausente em `df_sel` (não removemos),
porque a ausência em si é informação relevante (Problemas 4 e 9 — undercoverage da
PNAD durante a pandemia). A estratégia de tratamento (manter como `NaN` documentado
vs. imputar) é decidida e justificada na seção 9.

---

## 8. Limpeza e pré-processamento 

### 8.1 `periodo` (AAAASS / AAAA) → `ano` + `trimestre`

- Tabelas 4093/5436: `periodo` no formato `AAAASS` (ex.: `202001` = 2020, T1).
- Tabela 7322: `periodo` é só o ano (`2021`..`2024`) — não tem trimestre
  (indicador anual).

In [26]:
def _split_periodo(row):
    periodo = row["periodo"]
    if row["tabela_id"] == 7322:
        return pd.Series({"ano": periodo, "trimestre": pd.NA})
    return pd.Series({"ano": periodo // 100, "trimestre": periodo % 100})


df_sel[["ano", "trimestre"]] = df_sel.apply(_split_periodo, axis=1)
df_sel["ano"] = df_sel["ano"].astype(int)
df_sel["trimestre"] = df_sel["trimestre"].astype("Int64")  # nullable int (NA p/ tabela 7322)

df_sel = df_sel.drop(columns=["periodo"])

print(df_sel[["tabela_id", "ano", "trimestre"]].drop_duplicates().groupby("tabela_id").agg(
    {"ano": ["min", "max"], "trimestre": lambda x: sorted(x.dropna().unique().tolist())}
))
df_sel.head()

            ano           trimestre
            min   max      <lambda>
tabela_id                          
4093       2020  2025  [1, 2, 3, 4]
5436       2020  2025  [1, 2, 3, 4]
7322       2021  2024            []


,tabela_id,uf_nome,regiao_nome,sexo,nivel_instrucao,variavel_nome,unidade_medida,valor,ano,trimestre
16,4093,Paraná,NaN,Masculino,NaN,Pessoas de 14 anos ou mais de idade ocupadas na semana de referência,Mil pessoas,3210.0,2020,1
17,4093,Paraná,NaN,Feminino,NaN,Pessoas de 14 anos ou mais de idade ocupadas na semana de referência,Mil pessoas,2374.0,2020,1
56,4093,Paraná,NaN,Masculino,NaN,Taxa de informalidade das pessoas de 14 anos ou mais de idade ocupadas na se...,%,32.4,2020,1
57,4093,Paraná,NaN,Feminino,NaN,Taxa de informalidade das pessoas de 14 anos ou mais de idade ocupadas na se...,%,30.8,2020,1
84,4093,Paraná,NaN,Masculino,NaN,Pessoas de 14 anos ou mais de idade ocupadas na semana de referência,Mil pessoas,NaN,2020,2


### 8.2 `periodo_label` legível

In [27]:
df_sel["periodo_label"] = df_sel.apply(
    lambda r: f"{r['ano']}-T{r['trimestre']}" if pd.notna(r["trimestre"]) else str(r["ano"]),
    axis=1,
)
df_sel[["tabela_id", "ano", "trimestre", "periodo_label"]].drop_duplicates().head(8)

,tabela_id,ano,trimestre,periodo_label
16,4093,2020,1,2020-T1
84,4093,2020,2,2020-T2
152,4093,2020,3,2020-T3
220,4093,2020,4,2020-T4
288,4093,2021,1,2021-T1
356,4093,2021,2,2021-T2
424,4093,2021,3,2021-T3
492,4093,2021,4,2021-T4


### 8.3 Duplicidades

In [28]:
n_dup = df_sel.duplicated().sum()
print(f"Linhas duplicadas em df_sel: {n_dup}")

# Cada combinação (tabela_id, uf_nome/regiao_nome, periodo, sexo, nivel_instrucao,
# variavel_nome) deveria ser única — checagem da chave de granularidade
chave = ["tabela_id", "uf_nome", "regiao_nome", "ano", "trimestre", "sexo", "nivel_instrucao", "variavel_nome"]
n_dup_chave = df_sel.duplicated(subset=chave).sum()
print(f"Linhas duplicadas pela chave de granularidade: {n_dup_chave}")

Linhas duplicadas em df_sel: 0
Linhas duplicadas pela chave de granularidade: 0


### 8.4 Padronização de categorias de texto

`uf_nome`, `regiao_nome` e `nivel_instrucao` têm `NaN` para tabelas que não possuem
aquela dimensão (Problema 8). Substituímos por `"Não se aplica"` para deixar
explícito que a ausência é estrutural (não é um dado faltante a ser
imputado/investigado) — diferente do `NaN` em `valor`, que é tratado na seção 9.

In [29]:
for col in ["uf_nome", "regiao_nome", "nivel_instrucao"]:
    df_sel[col] = df_sel[col].fillna("Não se aplica")

# checagem de consistência de texto (sem variações de acentuação/caixa)
for col in ["uf_nome", "regiao_nome", "sexo", "nivel_instrucao", "tabela_id", "unidade_medida"]:
    print(f"{col}: {sorted(df_sel[col].astype(str).unique())}")

uf_nome: ['Não se aplica', 'Paraná', 'Rio Grande do Sul', 'Santa Catarina']
regiao_nome: ['Não se aplica', 'Sul']
sexo: ['Feminino', 'Masculino']
nivel_instrucao: ['Fundamental completo', 'Fundamental incompleto', 'Médio completo', 'Médio incompleto', 'Não se aplica', 'Sem instrução', 'Superior completo', 'Superior incompleto', 'Total']
tabela_id: ['4093', '5436', '7322']
unidade_medida: ['%', 'Mil pessoas', 'Reais']


---

## 9. Tratamento de valores faltantes 

### 9.1 Quantificação (antes do tratamento)

In [30]:
print(f"Total de NaN em 'valor': {df_sel['valor'].isna().sum()} / {len(df_sel)}\n")

print("NaN por tabela_id / variavel_nome:")
print(
    df_sel[df_sel["valor"].isna()]
    .groupby(["tabela_id", "variavel_nome"])
    .size()
    .to_frame("n_nan")
)

print("\nPeríodos afetados (todos em 4093):")
print(sorted(df_sel.loc[df_sel["valor"].isna(), "periodo_label"].unique()))

Total de NaN em 'valor': 96 / 448

NaN por tabela_id / variavel_nome:
                                                                                            n_nan
tabela_id variavel_nome                                                                          
4093      Pessoas de 14 anos ou mais de idade ocupadas na semana de referência                 48
          Taxa de informalidade das pessoas de 14 anos ou mais de idade ocupadas na sem...     48

Períodos afetados (todos em 4093):
['2020-T2', '2020-T3', '2020-T4', '2021-T1', '2021-T2', '2021-T3', '2021-T4', '2022-T1']


### 9.2 Estratégia de tratamento

Os 96 valores ausentes (todos em `tabela_id == 4093`, indicadores "Pessoas ocupadas"
e "Taxa de informalidade", trimestres 2020T2–2022T1) refletem **undercoverage real
da PNAD durante a pandemia** (Problemas 4 e 9, seção 5) — o IBGE optou por não
divulgar esses valores por insuficiência amostral, não por falha na coleta deste
projeto.

**Decisão:** manter `valor` como `NaN` (preserva a informação de que o dado *não
existe na fonte*, essencial para um relatório honesto sobre a pandemia). Para
demonstrar a técnica de imputação (exigência do PM3), criamos uma coluna adicional
`valor_imputado`, preenchida pela **média do grupo** (`uf_nome`, `sexo`,
`variavel_nome`) — ou seja, assume-se que o indicador naquele trimestre seguiu o
padrão médio da série da mesma UF/sexo/indicador. Ambas as colunas seguem para o
dataset final (seção 16), permitindo ao analista escolher qual usar conforme a
análise.

In [31]:
grupo = ["uf_nome", "sexo", "variavel_nome"]
df_sel["valor_imputado"] = df_sel["valor"].fillna(
    df_sel.groupby(grupo)["valor"].transform("mean")
)

print(f"NaN em 'valor':          {df_sel['valor'].isna().sum()}")
print(f"NaN em 'valor_imputado': {df_sel['valor_imputado'].isna().sum()}")

# Exemplo: trimestre 2020-T2, ocupados em PR/Masculino
df_sel[
    (df_sel["periodo_label"] == "2020-T2")
    & (df_sel["uf_nome"] == "Paraná")
    & (df_sel["sexo"] == "Masculino")
][["periodo_label", "uf_nome", "sexo", "variavel_nome", "valor", "valor_imputado"]]

NaN em 'valor':          96
NaN em 'valor_imputado': 0


,periodo_label,uf_nome,sexo,variavel_nome,valor,valor_imputado
84,2020-T2,Paraná,Masculino,Pessoas de 14 anos ou mais de idade ocupadas na semana de referência,NaN,3390.4375
124,2020-T2,Paraná,Masculino,Taxa de informalidade das pessoas de 14 anos ou mais de idade ocupadas na se...,NaN,32.6375


---

## 10. Tratamento de outliers

Análise via **IQR** (1.5×) de `valor_imputado`, separada por `tabela_id` +
`variavel_nome` (cada combinação tem sua própria escala/unidade — Problema 6).

In [32]:
def _flag_outliers(grupo_df):
    q1, q3 = grupo_df["valor_imputado"].quantile([0.25, 0.75])
    iqr = q3 - q1
    limite_inf, limite_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return (grupo_df["valor_imputado"] < limite_inf) | (grupo_df["valor_imputado"] > limite_sup)


df_sel["is_outlier"] = (
    df_sel.groupby(["tabela_id", "variavel_nome"], group_keys=False)
    .apply(_flag_outliers, include_groups=False)
)

print(f"Outliers detectados (IQR 1.5x): {df_sel['is_outlier'].sum()} / {len(df_sel)}\n")
print(df_sel[df_sel["is_outlier"]].groupby(["tabela_id", "variavel_nome"]).size())
df_sel[df_sel["is_outlier"]][
    ["tabela_id", "uf_nome", "regiao_nome", "periodo_label", "sexo", "nivel_instrucao", "variavel_nome", "valor_imputado"]
]

Outliers detectados (IQR 1.5x): 8 / 448

tabela_id  variavel_nome                      
7322       Pessoas de 10 anos ou mais de idade    8
dtype: int64


,tabela_id,uf_nome,regiao_nome,periodo_label,sexo,nivel_instrucao,variavel_nome,valor_imputado
5664,7322,Não se aplica,Sul,2021,Masculino,Total,Pessoas de 10 anos ou mais de idade,13001.0
5672,7322,Não se aplica,Sul,2021,Feminino,Total,Pessoas de 10 anos ou mais de idade,13382.0
5728,7322,Não se aplica,Sul,2022,Masculino,Total,Pessoas de 10 anos ou mais de idade,13128.0
5736,7322,Não se aplica,Sul,2022,Feminino,Total,Pessoas de 10 anos ou mais de idade,13453.0
5792,7322,Não se aplica,Sul,2023,Masculino,Total,Pessoas de 10 anos ou mais de idade,13219.0
5800,7322,Não se aplica,Sul,2023,Feminino,Total,Pessoas de 10 anos ou mais de idade,13599.0
5856,7322,Não se aplica,Sul,2024,Masculino,Total,Pessoas de 10 anos ou mais de idade,13219.0
5864,7322,Não se aplica,Sul,2024,Feminino,Total,Pessoas de 10 anos ou mais de idade,13755.0


**Interpretação e decisão:** os 8 "outliers" detectados pelo IQR são todos da
combinação `tabela_id == 7322` / `nivel_instrucao == "Total"` — ou seja, são a
**soma de todos os outros níveis de instrução** para aquele ano/sexo (≈13 mil, vs.
≈300–1.000 mil de cada nível individual). Não é um erro de digitação nem um valor
estatisticamente anômalo: é uma agregação legítima embutida na própria classificação
do IBGE (categoria "Total" dentro de "Nível de instrução"). **Decisão:** manter os
valores (não remover/transformar), mas a coluna `is_outlier` permanece no dataset
final para que análises que exijam apenas os subníveis (excluindo "Total") possam
filtrar com `is_outlier == False` ou, de forma mais explícita,
`nivel_instrucao != "Total"`. Para os indicadores das tabelas 4093/5436, **nenhum**
outlier foi detectado pelo IQR — a queda de rendimento/ocupação observada em
2020–2022 (seção 6) está dentro da variação normal da série.

---

## 11. Transformação dos dados

A separação de `periodo` em `ano` + `trimestre` + `periodo_label` já foi feita nas
seções 8.1/8.2 durante a limpeza (era pré-requisito dos gráficos temporais da seção
6). O que falta para completar a transformação é **agrupar `tabela_id` em rótulos
descritivos**, facilitando a leitura do dataset final por quem não conhece os
códigos do SIDRA.

In [33]:
TABELA_LABELS = {
    4093: "Força de trabalho e informalidade",
    5436: "Rendimento médio",
    7322: "Rendimento por nível de instrução",
}

df_sel["indicador_tabela"] = df_sel["tabela_id"].map(TABELA_LABELS)
df_sel[["tabela_id", "indicador_tabela"]].drop_duplicates()

,tabela_id,indicador_tabela
16,4093,Força de trabalho e informalidade
4896,5436,Rendimento médio
5664,7322,Rendimento por nível de instrução


---

## 12. Agregação de dados

Tabela agregada: rendimento médio anual (R$, `valor_imputado`, tabela 5436) por UF e
sexo — uma visão "anualizada" que resume os 4 trimestres de cada ano em um único
número, útil para um dashboard de BI que compare a evolução ano a ano sem o ruído da
sazonalidade trimestral.

In [34]:
rendimento_anual = (
    df_sel[df_sel["tabela_id"] == 5436]
    .groupby(["uf_nome", "sexo", "ano"])["valor_imputado"]
    .mean()
    .round(2)
    .reset_index()
    .rename(columns={"valor_imputado": "rendimento_medio_anual_r$"})
)
rendimento_anual

,uf_nome,sexo,ano,rendimento_medio_anual_r$
0,Paraná,Feminino,2020,2853.00
1,Paraná,Feminino,2022,2717.33
2,Paraná,Feminino,2023,2831.50
3,Paraná,Feminino,2024,3064.00
4,Paraná,Feminino,2025,3289.00
5,Paraná,Masculino,2020,3918.00
6,Paraná,Masculino,2022,3719.00
7,Paraná,Masculino,2023,3793.75
8,Paraná,Masculino,2024,4047.00
9,Paraná,Masculino,2025,4377.50


**Interpretação:** o ano de **2021 não aparece** na tabela — reflexo direto do
"buraco" temporal da tabela 5436 (Problema 9, 2020T2–2022T1): 2020 tem só o dado de
T1 e 2022 tem só T2–T4, então a média anual de 2021 ficaria baseada em 0 trimestres
reais (mesmo usando `valor_imputado`, não há nenhum trimestre de 2021 na base — a
imputação preenche valores ausentes *dentro* de uma linha existente, não cria linhas
para períodos inexistentes). Olhando a evolução de 2020 a 2025: o rendimento
masculino cresce de forma consistente nas três UFs (PR: R$ 3.918 → R$ 4.378, +11,7%;
RS: R$ 3.863 → R$ 4.274, +10,6%; SC: R$ 3.901 → R$ 4.636, +18,8%), enquanto o
feminino cresce a um ritmo semelhante ou maior em termos relativos (PR: +15,3%; RS:
+15,7%; SC: +19,4%). Como resultado, o gap percentual (Masc. vs. Fem.) recua
ligeiramente em PR (37,3% → 33,1%) e RS (37,6% → 31,6%), e fica praticamente estável
em SC (31,7% → 31,1%) — um sinal de leve convergência, mas o gap permanece acima de
30% em todos os casos até 2025.

---

## 13. Normalização e padronização

Aplicamos **Min-Max scaling** (`sklearn.preprocessing.MinMaxScaler`) sobre
`valor_imputado`, gerando `valor_normalizado` no intervalo `[0, 1]`. A escala é
ajustada **separadamente para cada combinação `(tabela_id, variavel_nome)`**, pois
cada indicador tem sua própria unidade/grandeza (Mil pessoas, %, R$ — Problema 6) —
normalizar todos juntos faria, por exemplo, percentuais (0-100) dominarem reais
(centenas/milhares) de forma artificial. Min-Max foi escolhido em vez de
`StandardScaler` (z-score) porque o objetivo é colocar todos os indicadores na
mesma escala `[0, 1]` para visualizações comparativas (ex.: um único gráfico com
ocupados, informalidade e rendimento na mesma escala), o que é mais intuitivo de
interpretar do que desvios-padrão.

In [35]:
from sklearn.preprocessing import MinMaxScaler


def _normalizar_grupo(grupo_df):
    valores = grupo_df[["valor_imputado"]]
    return pd.Series(
        MinMaxScaler().fit_transform(valores).ravel(), index=grupo_df.index
    )


df_sel["valor_normalizado"] = (
    df_sel.groupby(["tabela_id", "variavel_nome"], group_keys=False)
    .apply(_normalizar_grupo, include_groups=False)
)

# Antes vs. depois, para um indicador (rendimento médio, tabela 5436)
df_sel[df_sel["tabela_id"] == 5436][
    ["periodo_label", "uf_nome", "sexo", "valor_imputado", "valor_normalizado"]
].sort_values("valor_imputado").iloc[[0, -1]]

,periodo_label,uf_nome,sexo,valor_imputado,valor_normalizado
4913,2022-T2,Paraná,Feminino,2654.0,0.0
5376,2025-T3,Santa Catarina,Masculino,4676.0,1.0


---

## 14. Discretização dos dados 

Discretizamos o **rendimento médio** (`valor_imputado`, tabela 5436) em três faixas
— `baixo` / `médio` / `alto` — usando `pd.qcut` (tercis: cada faixa concentra ~1/3
das observações). Essa coluna `faixa_rendimento` é útil para BI (ex.: segmentar
UF/trimestres por faixa de rendimento sem precisar olhar o valor exato) e para
análises categóricas (ex.: contar quantos trimestres cada UF/sexo passou em cada
faixa). Para as demais tabelas (4093/7322), `faixa_rendimento` fica `"Não se
aplica"`.

In [36]:
df_sel["faixa_rendimento"] = "Não se aplica"

mask_5436 = df_sel["tabela_id"] == 5436
df_sel.loc[mask_5436, "faixa_rendimento"] = pd.qcut(
    df_sel.loc[mask_5436, "valor_imputado"],
    q=3,
    labels=["baixo", "médio", "alto"],
)

print(df_sel.loc[mask_5436, "faixa_rendimento"].value_counts())
df_sel.loc[mask_5436, ["periodo_label", "uf_nome", "sexo", "valor_imputado", "faixa_rendimento"]].sample(
    5, random_state=42
).sort_values("valor_imputado")

faixa_rendimento
alto     32
baixo    32
médio    32
Name: count, dtype: int64


,periodo_label,uf_nome,sexo,valor_imputado,faixa_rendimento
5505,2023-T3,Rio Grande do Sul,Feminino,2936.0,baixo
5153,2020-T1,Santa Catarina,Feminino,2961.0,baixo
5473,2023-T1,Rio Grande do Sul,Feminino,2983.0,baixo
5536,2024-T1,Rio Grande do Sul,Masculino,3940.0,alto
5648,2025-T4,Rio Grande do Sul,Masculino,4364.0,alto


---

## 15. Feature Engineering 

A principal feature derivada é o **gap salarial percentual** por UF e período —
formalizando a desigualdade observada nas seções 6.5/6.6/6.7:

```
gap_salarial_pct = (rendimento_masculino - rendimento_feminino) / rendimento_feminino * 100
```

Calculado via `pivot_table` de `sexo` (uma coluna para "Masculino" e outra para
"Feminino" do `valor_imputado`, indexado por `uf_nome` + `periodo_label`) e depois
reanexado a `df_sel` — ambas as linhas (Masculino e Feminino) de um mesmo
UF/período recebem o mesmo `gap_salarial_pct`, pois é uma medida do par, não de um
sexo isoladamente. As demais features de transformação/discretização (`ano`,
`trimestre`, `periodo_label`, `faixa_rendimento`, `is_outlier`,
`valor_normalizado`, `indicador_tabela`) já foram criadas nas seções 8, 11, 13 e
14 — `df_sel` consolida todas elas.

In [37]:
df_5436 = df_sel[df_sel["tabela_id"] == 5436]

pivot_sexo = df_5436.pivot_table(
    index=["uf_nome", "periodo_label"], columns="sexo", values="valor_imputado"
)
pivot_sexo["gap_salarial_pct"] = (
    (pivot_sexo["Masculino"] - pivot_sexo["Feminino"]) / pivot_sexo["Feminino"] * 100
).round(2)

df_sel = df_sel.merge(
    pivot_sexo["gap_salarial_pct"].reset_index(),
    on=["uf_nome", "periodo_label"],
    how="left",
)

df_sel.loc[df_sel["tabela_id"] == 5436, ["periodo_label", "uf_nome", "sexo", "valor_imputado", "gap_salarial_pct"]].head(6)

,periodo_label,uf_nome,sexo,valor_imputado,gap_salarial_pct
288,2020-T1,Paraná,Masculino,3918.0,37.33
289,2020-T1,Paraná,Feminino,2853.0,37.33
290,2022-T2,Paraná,Masculino,3630.0,36.77
291,2022-T2,Paraná,Feminino,2654.0,36.77
292,2022-T3,Paraná,Masculino,3747.0,39.19
293,2022-T3,Paraná,Feminino,2692.0,39.19


---

## 16. Consolidação dos datasets finais

`df_sel` reúne todas as colunas originais selecionadas (seção 7) mais as colunas
criadas nas seções 8–15: `ano`, `trimestre`, `periodo_label`, `indicador_tabela`,
`valor_imputado`, `is_outlier`, `valor_normalizado`, `faixa_rendimento`,
`gap_salarial_pct`. Como visto na seção 7, as tabelas 4093/5436 (UF, trimestral) e
7322 (Região, anual) nunca compartilharam granularidade geográfica/temporal — e
`gap_salarial_pct`/`faixa_rendimento` só existem para 5436. Por isso, em vez de um
único CSV com colunas que não se aplicam a parte das linhas, `df_sel` é dividido em
dois datasets finais, cada um só com as colunas que de fato usa:

- **`pnad_treated_data.csv`** — dataset **principal**: `tabela_id` 4093 + 5436,
  granularidade UF × sexo × trimestre. Remove `regiao_nome`/`nivel_instrucao`
  (sempre `"Não se aplica"` nessas linhas).
- **`pnad_context_data.csv`** — dataset de **contexto**: `tabela_id` 7322,
  granularidade Região × sexo × nível de instrução × ano. Remove
  `uf_nome`/`trimestre`/`faixa_rendimento`/`gap_salarial_pct` (nunca se aplicam a
  esta tabela).

Ambos são claramente diferentes do bronze (JSON bruto por tabela) e do silver
(`pnad_limpo.csv`, 5.920×15, sem as colunas derivadas).


In [38]:
config.gold_dir.mkdir(parents=True, exist_ok=True)

df_gap = df_sel[df_sel["tabela_id"].isin([4093, 5436])].drop(
    columns=["regiao_nome", "nivel_instrucao"]
)
df_contexto = df_sel[df_sel["tabela_id"] == 7322].drop(
    columns=["uf_nome", "trimestre", "faixa_rendimento", "gap_salarial_pct"]
)

gap_path = config.gold_dir / "pnad_treated_data.csv"
contexto_path = config.gold_dir / "pnad_context_data.csv"

df_gap.to_csv(gap_path, index=False, encoding="utf-8")
df_contexto.to_csv(contexto_path, index=False, encoding="utf-8")

print(f"principal (gap salarial): {df_gap.shape[0]} linhas x {df_gap.shape[1]} colunas")
print(f"colunas: {list(df_gap.columns)}")
print(f"salvo em: {gap_path}\n")

print(f"contexto (escolaridade): {df_contexto.shape[0]} linhas x {df_contexto.shape[1]} colunas")
print(f"colunas: {list(df_contexto.columns)}")
print(f"salvo em: {contexto_path}")


principal (gap salarial): 384 linhas x 15 colunas
colunas: ['tabela_id', 'uf_nome', 'sexo', 'variavel_nome', 'unidade_medida', 'valor', 'ano', 'trimestre', 'periodo_label', 'valor_imputado', 'is_outlier', 'indicador_tabela', 'valor_normalizado', 'faixa_rendimento', 'gap_salarial_pct']
salvo em: c:\Users\gabri\OneDrive\Documentos\GitHub\mensal_3_data_cleaning\mensal_3\dados\gold\pnad_treated_data.csv

contexto (escolaridade): 64 linhas x 13 colunas
colunas: ['tabela_id', 'regiao_nome', 'sexo', 'nivel_instrucao', 'variavel_nome', 'unidade_medida', 'valor', 'ano', 'periodo_label', 'valor_imputado', 'is_outlier', 'indicador_tabela', 'valor_normalizado']
salvo em: c:\Users\gabri\OneDrive\Documentos\GitHub\mensal_3_data_cleaning\mensal_3\dados\gold\pnad_context_data.csv


---

## 17. Catálogo de dados 

Catálogo completo das colunas de `pnad_treated_data.csv` — descrição, tipo,
exemplo, origem (original/derivada), tratamento aplicado e uso esperado — está em
[`docs/catalogo_dados.md`](../docs/catalogo_dados.md). O catálogo de
`pnad_context_data.csv` está em
[`docs/catalogo_dados_contexto.md`](../docs/catalogo_dados_contexto.md).

---

## 18. DataOps e organização do projeto 

- Estrutura de pastas, pipeline de execução (`01_coleta` → `02_silver` →
  `03_tratamento_pm3`) e convenções estão documentadas no `README.md` da raiz do
  projeto.
- `dados/bronze/` (JSON original do SIDRA) e `dados/gold/pnad_treated_data.csv` +
  `dados/gold/pnad_context_data.csv` (datasets finais) são versionados no
  repositório como evidência/entregável do PM3 (ver `.gitignore`).
